In [12]:
import pandas
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification,BertTokenizer, BertForSequenceClassification
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding

dataset_path = r"../../datasets/sentiment_dataset.csv"

In [13]:
dataset = pandas.read_csv(dataset_path)

In [14]:
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset["label"])

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [15]:
train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 3525
})

In [16]:
checkpoint = "distilbert-base-uncased"
checkpoint = "vinai/phobert-base" 
## https://huggingface.co/vinai/phobert-base 
# Lưu ý: vinai/phobert-base phải max_length = 256 

In [17]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
# Ensure consistent label names for training + deploy
model.config.id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
model.config.label2id = {'NEGATIVE': 0, 'POSITIVE': 1}
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28151.61it/s]
RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok i

In [18]:
# PhoBERT (RoBERTa-style) requires fixed max_length to avoid shape mismatches.
max_length = 256

def tokenizer_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
        return_token_type_ids=False,
    )


def preprocessing(ds):
    ds = ds.map(tokenizer_fn, batched=True, remove_columns=["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

In [19]:
print(train_dataset['text'][0])

 Đến lần muốn đến thêm lần ngon tuyệt


In [20]:
tokenized_train = preprocessing(train_dataset)
tokenized_test = preprocessing(test_dataset)

Map: 100%|██████████| 882/882 [00:00<00:00, 1885.83 examples/s]


In [21]:
import numpy as np
import torch
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds)
        , 'f1': f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir='../../models/save/bert-finetuned-negative-positive',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    # warmup_ratio=0.06,
    warmup_steps = 0.06 * len(tokenized_train) // 16,  # 6% of training steps
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    disable_tqdm=True,
    logging_steps=50,
    seed=42,
    report_to='none',
)

from transformers import ProgressCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[ProgressCallback()],
)


In [22]:
train_result = trainer.train()
metrics = trainer.evaluate()
print(metrics)


  8%|▊         | 51/663 [00:10<02:05,  4.89it/s]

{'loss': '0.585', 'grad_norm': '8.3', 'learning_rate': '1.895e-05', 'epoch': '0.2262'}
{'loss': '0.585', 'grad_norm': '8.3', 'learning_rate': '1.895e-05', 'epoch': '0.2262'}


 15%|█▌        | 101/663 [00:20<01:48,  5.19it/s]

{'loss': '0.2826', 'grad_norm': '3.468', 'learning_rate': '1.742e-05', 'epoch': '0.4525'}
{'loss': '0.2826', 'grad_norm': '3.468', 'learning_rate': '1.742e-05', 'epoch': '0.4525'}


 23%|██▎       | 151/663 [00:30<01:45,  4.86it/s]

...

{'eval_loss': '0.2105', 'eval_accuracy': '0.9331', 'eval_f1': '0.9454', 'eval_runtime': '2.751', 'eval_samples_per_second': '320.6', 'eval_steps_per_second': '10.18', 'epoch': '3'}
{'eval_loss': '0.2105', 'eval_accuracy': '0.9331', 'eval_f1': '0.9454', 'eval_runtime': '2.751', 'eval_samples_per_second': '320.6', 'eval_steps_per_second': '10.18', 'epoch': '3'}


100%|██████████| 663/663 [02:38<00:00,  5.76it/s]

{'train_runtime': '158.7', 'train_samples_per_second': '66.65', 'train_steps_per_second': '4.179', 'train_loss': '0.2163', 'epoch': '3'}
{'train_runtime': '158.7', 'train_samples_per_second': '66.65', 'train_steps_per_second': '4.179', 'train_loss': '0.2163', 'epoch': '3'}


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '0.2105', 'eval_accuracy': '0.9331', 'eval_f1': '0.9454', 'eval_runtime': '3.021', 'eval_samples_per_second': '291.9', 'eval_steps_per_second': '9.268', 'epoch': '3'}
{'eval_loss': 0.21047520637512207, 'eval_accuracy': 0.9331065759637188, 'eval_f1': 0.9454209065679926, 'eval_runtime': 3.0213, 'eval_samples_per_second': 291.928, 'eval_steps_per_second': 9.268, 'epoch': 3.0}


In [23]:
finetuned_model_path = '../../output/bert-finetuned-negative-positive'

In [24]:
token_save = tokenizer.save_pretrained(finetuned_model_path)
# trained = trainer.save_model("./bert-finetuned-negative-positive")
trained = model.save_pretrained(finetuned_model_path)

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.96s/it]
